# 04 — Construção dos perfis documentais para Coverage dos modelos Qwen

Este notebook gera as evidências textuais por documento utilizadas posteriormente no cálculo de `Coverage` das abordagens Qwen.

A lógica reutilizável está em `src/preprocessing/build_document_profiles_qwen.py`.

Para cada autor e documento são preservadas, separadamente:

- o título;
- cada palavra-chave;
- o resumo.

A saída tem a estrutura:

`autor -> documento -> lista de unidades textuais`

Essa representação é distinta de `perfis_estruturados_qwen.json`. Ela não é utilizada como entrada do LLM e não realiza inferência ou cálculo de métricas.

## 1. Configuração do projeto

Os caminhos onde estão os arquivos processados da coleção.

In [ ]:
from pathlib import Path
import json
import sys

current = Path.cwd().resolve()
candidates = [current, *current.parents]
PROJECT_ROOT = next((p for p in candidates if (p / "src").exists()), current)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing.build_document_profiles_qwen import (
    carregar_autores_qrels,
    gerar_perfis_documento_qwen,
)

print(f"Raiz do projeto: {PROJECT_ROOT}")

## 2. Caminhos

In [ ]:
DATA_DIR = PROJECT_ROOT / "data" / "processed"

DOCUMENTS = DATA_DIR / "filtered_documents.json"
QRELS = DATA_DIR / "ground_truth" / "LExR-prof-qrels_filtrado"
OUTPUT = DATA_DIR / "perfis_documento_qwen.json"

for name, path in {
    "filtered_documents.json": DOCUMENTS,
    "LExR-prof-qrels_filtrado": QRELS,
}.items():
    print(f"{name:32s} -> {'OK' if path.exists() else 'não encontrado'}")

## 3. Seleção dos autores

Os IDs da primeira coluna dos qrels filtrados definem o conjunto de autores considerado nesta etapa.

In [ ]:
autores_alvo = carregar_autores_qrels(str(QRELS))
print(f"Autores nos qrels: {len(autores_alvo):,}")

## 4. Construção das evidências por documento

Cada documento é associado a todos os coautores que pertencem ao conjunto avaliado.

A limpeza remove HTML, entidades e caracteres considerados ruído, preservando caixa, acentuação e pontuação básica.

In [ ]:
perfis_documento = gerar_perfis_documento_qwen(
    str(DOCUMENTS),
    autores_alvo,
)

OUTPUT.parent.mkdir(parents=True, exist_ok=True)
with OUTPUT.open("w", encoding="utf-8") as f:
    json.dump(perfis_documento, f, ensure_ascii=False)

print(f"Arquivo salvo em: {OUTPUT}")

## 5. Resumo e verificação

In [ ]:
total_docs = sum(len(docs) for docs in perfis_documento.values())
total_partes = sum(
    len(partes)
    for docs in perfis_documento.values()
    for partes in docs.values()
)

print(f"Autores com documentos: {len(perfis_documento):,}")
print(f"Entradas autor × documento: {total_docs:,}")
print(f"Unidades textuais: {total_partes:,}")

for author_id, docs in list(perfis_documento.items())[:2]:
    print("\n" + "=" * 80)
    print(f"Autor: {author_id}")
    for doc_id, partes in list(docs.items())[:1]:
        print(f"Documento: {doc_id}")
        print(json.dumps(partes, ensure_ascii=False, indent=2))

## 6. Arquivo produzido

O notebook gera:

- `perfis_documento_qwen.json`

Esse arquivo é utilizado pela etapa de avaliação para verificar quantas publicações de cada pesquisador são representadas pelas tags recuperadas.

Nenhuma tag é gerada neste notebook e nenhuma métrica é calculada aqui.